In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import pchip_interpolate

from igs_ga.optimization.modo import Preferendus

In [ ]:
# Define preference functions for the two objectives
def p_shopping_value(x1, x2):
    """Objective to maximize the shopping added value

    :param x1: 1st design variable
    :param x2: 2nd design variable
    """
    normalized_x1 = 1 - (x1 - 100) / (1000 - 100)
    normalized_x2 = (x2 - 800) / (30_000 - 800)

    ret = 100 * (np.sqrt(normalized_x1 ** 2 + normalized_x2) ** 2)

    if isinstance(ret, np.ndarray):
        mask = ret > 100
        ret[mask] = 0
    else:
        if ret > 100:
            ret = 0

    return ret

def p_sustainability(x1, x2):
    """Objective that describes the relative sustainability.

    :param x1: 1st design variable
    :param x2: 2nd design variable
    """
    return pchip_interpolate([0, 1.2, 2], [0, 100, 60], x2 / (20_000 * x1 / 400))

In [ ]:
# Set weights for the different objectives
w1 = 0.65
w2 = 0.35

# Define stakeholder preferences
stakeholder_preferences = {
    1: {
        "1_shopping": lambda x: x if 0 <= x <= 100 else (0 if x > 100 else 0),
        "2_sustainability": lambda x: x if 0 <= x <= 100 else (100 if x > 100 else 0),
    },
}

# Set objective weights
objective_weights = {
    1: {"1_shopping": w1, "2_sustainability": w2},
}

In [ ]:
# Set optimization options
optimization_options = {
    'population_size': 375,
    'generations': 10,
    'mutation_rate': 0.80,
    'max_stall': 10,
    'hall_of_fame_size':375
}

@Preferendus(
    preference_functions=stakeholder_preferences,
    stakeholder_weights=[1.0],
    variable_types=[float, int],
    value_range=[(100.0, 1000.0), (800, 30000)],
    objective_weights=objective_weights,
    optimization_options=optimization_options
)
def run_simulation(x):
    """Objective function that calculates the two objectives for the supermarket problem.

    :param x: Design variables [x1, x2] where x1 is distance in meters and x2 is number of products
    :return: None, None, plus the two objective values
    """
    # Calculate the shopping value preference
    shopping_value = p_shopping_value(x[0], x[1])

    # Calculate the sustainability preference
    sustainability = p_sustainability(x[0], x[1])

    return None, None, shopping_value, sustainability
# Run the optimization
initial_guess = [500, 15000]  # Initial guess in the middle of the bounds
result = run_simulation(initial_guess)

# Extract optimal design variables
design_variables = result['x_opt']

print(f'Optimal result for a distance of {round(design_variables[0], 2)} meters and '
      f'{design_variables[1]} products')
print(f"Utility: {result['utility']}")

In [ ]:
# Create figure that shows the results in the solution space
fig, ax = plt.subplots(figsize=(7, 7))
ax.set_xlim((0, 1200))
ax.set_ylim((600, 32_000))
ax.set_xlabel('x1 [m]')
ax.set_ylabel('x2 [-]')
ax.set_title('Solution space')

# Define corner points of solution space
x_fill = [100, 1000, 1000, 100]
y_fill = [30_000, 30_000, 800, 800]

ax.fill_between(x_fill, y_fill, color='#539ecd', label='Solution space')
ax.scatter(design_variables[0], design_variables[1], label='Optimal solution MODO', color='tab:purple')

ax.grid()  # show grid
fig.legend()  # show legend

# Create arrays for plotting continuous preference curves
c1 = np.linspace(0, 2, 100)
c2 = np.linspace(0, 2, 100)

# Calculate the preference functions
norm_1 = np.linspace(0, 1, 100)
norm_2 = np.linspace(0, 1, 100)
N1, N2 = np.meshgrid(norm_1, norm_2)

p1 = 100 * (np.sqrt(N1 ** 2 + N2) ** 2)
m = p1 > 100
p1[m] = 0
p2 = pchip_interpolate([0, 1.2, 2], [0, 100, 60], c2)

# Calculate individual preference scores for the results to plot them on the preference curves
normalized_c1x1 = 1 - (design_variables[0] - 100) / (1000 - 100)
normalized_c1x2 = (design_variables[1] - 800) / (30_000 - 800)

c2_res = design_variables[1] / (20_000 * design_variables[0] / 400)

p1_res = p_shopping_value(design_variables[0], design_variables[1])
p2_res = p_sustainability(design_variables[0], design_variables[1])

# Create figure that plots all preference curves and the preference scores of the returned results
fig = plt.figure()
ax2 = fig.add_subplot(1, 1, 1)
ax2.plot(c2, p2, label='Preference curve')
ax2.scatter(c2_res, p2_res, label='Optimal solution MODO', color='tab:purple')
ax2.set(xlabel='Sustainability index', ylabel='Preference')
ax2.set_title('Preference Curve Transport Sustainability & Wasted')
ax2.legend()
ax2.grid()

fig = plt.figure()
ax1 = fig.add_subplot(1, 1, 1, projection='3d')
surf = ax1.plot_surface(N1, N2, p1, label='Preference curve')
ax1.scatter(normalized_c1x1, normalized_c1x2, p1_res, label='Optimal solution MODO', color='tab:purple')
ax1.set(xlabel='Normalized travel distance', ylabel='Normalized items in assortiment', zlabel='Preference')
ax1.set_title('Preference Curves Shopping Added Value')
ax1.view_init(elev=15, azim=160)
surf._facecolors2d = surf._facecolor3d
surf._edgecolors2d = surf._edgecolor3d
ax1.legend()
ax1.grid()

fig = plt.figure()
ax3 = fig.add_subplot(1, 1, 1)
img = ax3.imshow(p1, cmap='GnBu', interpolation='nearest')
ax3.scatter(normalized_c1x1 * 100, normalized_c1x2 * 100, label='Optimal solution MODO', color='tab:purple')
ax3.set(xlabel='Normalized travel distance', ylabel='Normalized items in assortiment')
ax3.set_title('Preference Curves Shopping Added Value')
ax3.legend()
ax3.grid()

plt.colorbar(img, ax=ax3, location='left')

plt.show()